In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import spacy

In [ ]:
data = pd.read_csv("../DATA/quran_en_ar_data.csv")
data.head()

In [ ]:

# Extract all entities in the passage/surah
nlp = spacy.load("en_core_web_sm")
def extract_entities(text: str, entity_type:str = "PERSON"):
  doc = nlp(text)
  entities = [ent.text for ent in doc.ents if ent.label_ == entity_type]
  return entities

In [ ]:
extract_entities("Abraham was a son of Terah")

In [ ]:
data["person"] = data["text"].apply(extract_entities)
data[["text","person"]]

In [ ]:
data["person"].tolist()

In [ ]:

# Wordcloud plot: The most commonest/frequent names
def plot_wordcloud(docx):
  my_wordcloud = WordCloud().generate(docx)
  plt.imshow(my_wordcloud, interpolation="bilinear")
  plt.axis("off")
  plt.show()

In [ ]:
person_list = [i for item in data['person'].tolist() for i in item]
person_list

In [ ]:
" ".join(person_list)

In [ ]:
plot_wordcloud(" ".join(person_list))

In [ ]:

# Find the most commonest names or character
from collections import Counter
commonest_names = Counter(person_list)
commonest_names.most_common(30)

In [ ]:
common_quran_names_data = pd.DataFrame(commonest_names.most_common(30),columns=['names','count'])
common_quran_names_data

In [ ]:
import plotly.express as px
px.bar(common_quran_names_data,x="names",y="count")
data["person"]

In [ ]:
# data.to_csv("../DATA/quran_characters.csv")

In [ ]:
clean_data = data[data["person"].map(len) > 0]
clean_data.head()

In [ ]:
clean_data["person"] = clean_data["person"].apply(lambda x: [item.split()[0] for item in x])
clean_data.shape

In [ ]:

def get_relationships(clean_df,window_size:int = 5, entity_column:str ="person"):
  relationships = []

  for i in range(clean_df.index[-1]):
    end_i = min(i+5, clean_df.index[-1])
    char_list = sum((clean_df.loc[i: end_i][entity_column]), [])
    
    # Remove duplicated characters that are next to each other
    char_unique = [char_list[i] for i in range(len(char_list)) 
                    if (i==0) or char_list[i] != char_list[i-1]]
    
    if len(char_unique) > 1:
      for idx, a in enumerate(char_unique[:-1]):
        b = char_unique[idx + 1]
        relationships.append({"source": a, "target": b})
  return relationships

In [ ]:
relationship_data = get_relationships(clean_data)
relationship_data

In [ ]:
relationship_data = pd.DataFrame(relationship_data)
relationship_data.head()

In [ ]:
import numpy as np
relationship_data = pd.DataFrame(np.sort(relationship_data.values, axis = 1), columns = relationship_data.columns)
relationship_data

In [ ]:
relationship_data["value"] = 1
relationship_data = relationship_data.groupby(["source","target"],sort=False,as_index=False).sum()
relationship_data

In [ ]:

# Create the network graph
import networkx as nx
quran_graph = nx.from_pandas_edgelist(relationship_data,source="source",target="target",edge_attr="value",create_using=nx.Graph())

In [ ]:
# general connections
node_degree = dict(quran_graph.degree())
node_degree

In [ ]:
import visualizer
# https://gist.github.com/mogproject/50668d3ca60188c50e6ef3f5f3ace101
def plot_network(my_graph, **kwargs):
  # Set default values for keyword arguments
  default_kwargs = {
    "node_size": 10,
    "node_border_width": 1,
    "edge_width": 0.5,
    "node_color": "blue",
  }
  # Update default_kwargs with the provided kwargs
  default_kwargs.update(kwargs)

  pos = nx.spring_layout(my_graph, iterations=20)
  vis = visualizer.GraphVisualization(my_graph, pos, **default_kwargs)
  # vis = visualizer.Visualizer(my_graph,pos,**default_kwargs)
  # vis = visualizer. 
  fig = vis.create_figure(height=800, width=800, showlabel=False)
  return fig

In [ ]:
# plot_network(quran_graph)

In [ ]:

# Function to create node color list based on communities
color_map_lists = ['Blues', 'BrBG', 'BuGn', 'BuPu', 'CMRmap', 'GnBu', 'Greens', 'Greys', 'OrRd', 'Oranges', 'PRGn', 'PiYG', 'PuBu', 'PuBuGn', 'PuOr', 'PuRd', 'Purples', 'RdBu', 'RdGy', 'RdPu', 'RdYlBu', 'RdYlGn', 'Reds', 'Spectral', 'Wistia', 'YlGn', 'YlGnBu', 'YlOrBr', 'YlOrRd', 'afmhot', 'autumn', 'binary', 'bone', 'brg', 'bwr', 'cool', 'coolwarm', 'copper', 'cubehelix', 'flag', 'gist_earth', 'gist_gray', 'gist_heat', 'gist_ncar', 'gist_rainbow', 'gist_stern', 'gist_yarg', 'gnuplot', 'gnuplot2', 'gray', 'hot', 'hsv', 'jet', 'nipy_spectral', 'ocean', 'pink', 'prism', 'rainbow', 'seismic', 'spring', 'summer', 'terrain', 'winter', 'Accent', 'Dark2', 'Paired', 'Pastel1', 'Pastel2', 'Set1', 'Set2', 'Set3', 'tab10', 'tab20', 'tab20b', 'tab20c']
color_map_list = ['aliceblue', 'antiquewhite', 'aqua', 'aquamarine', 'azure', 'beige', 'bisque', 'black', 'blanchedalmond', 'blue', 'blueviolet', 'brown', 'burlywood', 'cadetblue', 'chartreuse', 'chocolate', 'coral', 'cornflowerblue', 'cornsilk', 'crimson', 'cyan', 'darkblue', 'darkcyan', 'darkgoldenrod', 'darkgray', 'darkgrey', 'darkgreen', 'darkkhaki', 'darkmagenta', 'darkolivegreen', 'darkorange', 'darkorchid', 'darkred', 'darksalmon', 'darkseagreen', 'darkslateblue', 'darkslategray', 'darkslategrey', 'darkturquoise', 'darkviolet', 'deeppink', 'deepskyblue', 'dimgray', 'dimgrey', 'dodgerblue', 'firebrick', 'floralwhite', 'forestgreen', 'fuchsia', 'gainsboro', 'ghostwhite', 'gold', 'goldenrod', 'gray', 'grey', 'green', 'greenyellow', 'honeydew', 'hotpink', 'indianred', 'indigo', 'ivory', 'khaki', 'lavender', 'lavenderblush', 'lawngreen', 'lemonchiffon', 'lightblue', 'lightcoral', 'lightcyan', 'lightgoldenrodyellow', 'lightgray', 'lightgrey', 'lightgreen', 'lightpink', 'lightsalmon', 'lightseagreen', 'lightskyblue', 'lightslategray', 'lightslategrey', 'lightsteelblue', 'lightyellow', 'lime', 'limegreen', 'linen', 'magenta', 'maroon', 'mediumaquamarine', 'mediumblue', 'mediumorchid', 'mediumpurple', 'mediumseagreen', 'mediumslateblue', 'mediumspringgreen', 'mediumturquoise', 'mediumvioletred', 'midnightblue', 'mintcream', 'mistyrose', 'moccasin', 'navajowhite', 'navy', 'oldlace', 'olive', 'olivedrab', 'orange', 'orangered', 'orchid', 'palegoldenrod', 'palegreen', 'paleturquoise', 'palevioletred', 'papayawhip', 'peachpuff', 'peru', 'pink', 'plum', 'powderblue', 'purple', 'red', 'rosybrown', 'royalblue', 'rebeccapurple', 'saddlebrown', 'salmon', 'sandybrown', 'seagreen', 'seashell', 'sienna', 'silver', 'skyblue', 'slateblue', 'slategray', 'slategrey', 'snow', 'springgreen', 'steelblue', 'tan', 'teal', 'thistle', 'tomato', 'turquoise', 'violet', 'wheat', 'white', 'whitesmoke', 'yellow', 'yellowgreen']
def create_community_node_colors(graph, communities):
  number_of_colors = len(communities)
  node_colors = [color_map_list[i] for i in range(number_of_colors)]
  community_map = {node: i for i, comm in enumerate(communities) for node in comm}
  return [node_colors[community_map[node]] for node in graph.nodes()]

In [ ]:
# Get communities with our graph
communities = nx.community.louvain_communities(quran_graph)

# Check num of communities in the graph
len(communities)

In [ ]:
node_colors = create_community_node_colors(quran_graph,communities)
# add the communities as an attribute
nx.set_node_attributes(quran_graph,communities,"group")

In [ ]:
node_degree

In [ ]:
# Add the degree as size to our graph as an attribute
nx.set_node_attributes(quran_graph,node_degree,"size")

#  Shortest path between two characters in the passage
nx.shortest_path(quran_graph,"Soul","Tasnim")